# Modelos Baseline — Modelos Simples de Referencia

**Objetivo:** Evaluar 3 modelos simples (Buy & Hold, Naive, Zero Prediction) para las 16 
combinaciones de ventanas temporales. Estos resultados establecen el suelo mínimo de rendimiento 
que cualquier red neuronal debe superar para justificar su complejidad.

**Métrica:** MAE (Mean Absolute Error)

In [1]:
import sys
import os
sys.path.append('..')
os.chdir('..')

## 1. Imports y configuración

In [2]:
import numpy as np
import pandas as pd

# Infraestructura compartida del proyecto
from src.data_pipeline import load_data, get_train_test
from src.evaluation import compute_mae, load_all_results
from src.baselines import (
    predict_buy_and_hold,
    predict_naive,
    predict_zero,
    evaluate_all_baselines
)
from config import INPUT_WINDOWS, OUTPUT_WINDOWS

## 2. Carga de datos

In [3]:
returns = load_data()
print(f"Shape: {returns.shape}")

[*********************100%***********************]  23 of 23 completed


Datos cargados: 16194 días, 23 activos
Rango: 1962-01-03 → 2026-05-07
Shape: (16194, 23)


## 3. Descripción de los modelos baseline

Se implementan 3 modelos simples de referencia:

- **Buy & Hold:** Predice la media histórica incondicional de los log-returns del conjunto de 
  entrenamiento. Asume que el mercado se comportará en el futuro como lo ha hecho en promedio 
  históricamente. Es el modelo simple requerido explícitamente por el enunciado.

- **Naive (Last Value):** Predice que el promedio futuro de los log-returns será igual al último 
  valor observado en la ventana de entrada. Es el benchmark más básico en series temporales: 
  "mañana será como hoy".

- **Zero Prediction:** Predice que el promedio futuro de los log-returns será cero. Los 
  log-returns diarios tienen media muy cercana a cero, lo que convierte a este modelo en un 
  baseline sorprendentemente competitivo, especialmente en horizontes cortos.

Ninguno de estos modelos tiene parámetros entrenables. No hay curvas de convergencia porque 
no existe proceso de entrenamiento.

## 4. Evaluación de baselines para las 16 combinaciones de ventanas

Se ejecutan los 3 baselines para cada combinación de ventanas de entrada (5, 10, 30, 90) 
y salida (1, 5, 30, 90), generando 48 resultados (3 modelos × 16 combinaciones). Todos se 
guardan automáticamente en el CSV compartido (`results/tables/all_results.csv`).

In [4]:
evaluate_all_baselines(returns)

Evaluando baselines para todas las combinaciones de ventanas...
Ventana entrada=5, salida=1
  X_train: (14570, 5, 23) | y_train: (14570, 23)
  X_test:  (1619, 5, 23)  | y_test:  (1619, 23)
Resultados guardados: Buy_and_Hold | in=5 out=1 | MAE test=0.012260
Resultados guardados: Naive_LastValue | in=5 out=1 | MAE test=0.017813
Resultados guardados: Zero_Prediction | in=5 out=1 | MAE test=0.012273
Ventana entrada=5, salida=5
  X_train: (14566, 5, 23) | y_train: (14566, 23)
  X_test:  (1619, 5, 23)  | y_test:  (1619, 23)
Resultados guardados: Buy_and_Hold | in=5 out=5 | MAE test=0.005589
Resultados guardados: Naive_LastValue | in=5 out=5 | MAE test=0.013677
Resultados guardados: Zero_Prediction | in=5 out=5 | MAE test=0.005610
Ventana entrada=5, salida=30
  X_train: (14544, 5, 23) | y_train: (14544, 23)
  X_test:  (1616, 5, 23)  | y_test:  (1616, 23)
Resultados guardados: Buy_and_Hold | in=5 out=30 | MAE test=0.002320
Resultados guardados: Naive_LastValue | in=5 out=30 | MAE test=0.012529

## 5. Tabla resumen de resultados

Se presentan los resultados de los 3 baselines organizados por combinación de ventanas. 
El MAE en test es la métrica que se usará para comparar contra los modelos de redes neuronales.

In [5]:
# Cargar todos los resultados guardados
results = load_all_results()

# Filtrar solo baselines
baselines_df = results[results['model_type'] == 'baseline'].copy()

# Tabla pivotada: MAE test por modelo y combinación de ventanas
baselines_df['window_combo'] = baselines_df.apply(
    lambda r: f"in={int(r['input_window'])}, out={int(r['output_window'])}", axis=1
)

pivot = baselines_df.pivot_table(
    index='window_combo',
    columns='model_name',
    values='mae_test'
)

# Ordenar las filas lógicamente
window_order = [f"in={iw}, out={ow}" for iw in INPUT_WINDOWS for ow in OUTPUT_WINDOWS]
pivot = pivot.reindex(window_order)

print("MAE Test por modelo y combinación de ventanas:")
print("=" * 65)
print(pivot.to_string())

MAE Test por modelo y combinación de ventanas:
model_name     Buy_and_Hold  Naive_LastValue  Zero_Prediction
window_combo                                                 
in=5, out=1        0.012260         0.017813         0.012273
in=5, out=5        0.005589         0.013677         0.005610
in=5, out=30       0.002320         0.012529         0.002355
in=5, out=90       0.001268         0.012240         0.001324
in=10, out=1       0.012260         0.017813         0.012273
in=10, out=5       0.005591         0.013681         0.005612
in=10, out=30      0.002320         0.012529         0.002355
in=10, out=90      0.001268         0.012240         0.001324
in=30, out=1       0.012268         0.017823         0.012280
in=30, out=5       0.005594         0.013685         0.005615
in=30, out=30      0.002320         0.012527         0.002355
in=30, out=90      0.001269         0.012233         0.001325
in=90, out=1       0.012282         0.017847         0.012295
in=90, out=5       0.00

## 6. Mejor baseline por combinación de ventanas

Para cada combinación de ventanas, se identifica qué modelo baseline obtiene el menor MAE 
en test. Este será el suelo que cada red neuronal deberá superar.

In [6]:
# Identificar el mejor baseline por combinación
best_per_combo = baselines_df.loc[
    baselines_df.groupby(['input_window', 'output_window'])['mae_test'].idxmin()
][['input_window', 'output_window', 'model_name', 'mae_test']].reset_index(drop=True)

best_per_combo.columns = ['Input Window', 'Output Window', 'Mejor Baseline', 'MAE Test']
print("Mejor baseline por combinación:")
print("=" * 65)
print(best_per_combo.to_string(index=False))

Mejor baseline por combinación:
 Input Window  Output Window Mejor Baseline  MAE Test
            5              1   Buy_and_Hold  0.012260
            5              5   Buy_and_Hold  0.005589
            5             30   Buy_and_Hold  0.002320
            5             90   Buy_and_Hold  0.001268
           10              1   Buy_and_Hold  0.012260
           10              5   Buy_and_Hold  0.005591
           10             30   Buy_and_Hold  0.002320
           10             90   Buy_and_Hold  0.001268
           30              1   Buy_and_Hold  0.012268
           30              5   Buy_and_Hold  0.005594
           30             30   Buy_and_Hold  0.002320
           30             90   Buy_and_Hold  0.001269
           90              1   Buy_and_Hold  0.012282
           90              5   Buy_and_Hold  0.005602
           90             30   Buy_and_Hold  0.002322
           90             90   Buy_and_Hold  0.001271


## 7. Análisis

Observaciones clave a extraer de los resultados:

1. **¿Qué baseline domina?** Identificar si Zero Prediction, Naive o Buy & Hold es 
   consistentemente mejor y en qué tipos de ventanas.

2. **Efecto de la ventana de salida:** El MAE tiende a cambiar significativamente con la 
   ventana de salida (1d vs 90d), ya que promediar más días reduce la varianza de los 
   log-returns.

3. **Efecto de la ventana de entrada:** Para Zero y Buy & Hold, la ventana de entrada tiene 
   impacto mínimo. Para Naive, sí influye porque la predicción depende del último valor 
   observado.

4. **Referencia para redes neuronales:** Cualquier modelo de red neuronal que no bata al 
   mejor baseline para una combinación dada no está justificando su complejidad ni su coste 
   computacional.